# Step 1: data, the exam, two baselines

Run `uv run python scripts/download_goodbooks.py` once, then run all cells (from the lab folder: `uv run jupyter lab`). Outputs are cleared before committing.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from folio_lab.data import load_shelf, load_books, load_ratings, match_to_goodbooks
shelf, books, ratings = load_shelf(), load_books(), load_ratings()
shelf.shelf.value_counts(), shelf.rating.value_counts().sort_index()

## Look at the data first (Tukey's exploratory data analysis)

When did I read and add books? Recent taste lives mostly in want-to-reads.

In [ ]:
pd.crosstab(shelf.date_added.dt.year, shelf.shelf)

## How much of my shelf is in goodbooks?

Matching ladder, strictest first: Goodreads id, ISBN-10, title + author surname.

In [ ]:
m = match_to_goodbooks(shelf, books)
liked = m[m.rating >= 4]
print(f"liked in goodbooks: {liked.gb_id.notna().sum()} of {len(liked)}")
m.matched_by.value_counts(dropna=False)

In [ ]:
want = m[m.shelf == 'to-read']
want.groupby(want.date_added.dt.year).gb_id.agg(lambda s: f'{s.notna().sum()} of {len(s)}')

## Is goodbooks' ratings file really in time order?

If it is, books published in 2016–17 can't be rated early in the file.

In [ ]:
y = ratings.merge(books[['book_id', 'original_publication_year']])
y.groupby(y.original_publication_year >= 2016).seq.agg(['min', 'mean', 'count'])

## The exam

10,000 readers with at least 10 liked books. Hide each one's last liked book (and, in a second run, the last 5); the model sees only what came before. Plus my own 48 liked books, each hidden in turn with only the shelf from before it visible.

In [ ]:
from folio_lab.splits import eligible_readers, leave_last_out
split = leave_last_out(ratings, eligible_readers(ratings), 1)
case = split.cases[0]
print(len(split.cases), 'readers; first reader sees', len(case.history), 'books; hidden:', case.targets)

## Baselines and the results table

`scripts/run_eval.py` scores every model on every run and writes `results/results.md`.

In [ ]:
from folio_lab.baselines import MostPopular
mp = MostPopular(); mp.fit(ratings, books)
books.set_index('book_id').loc[mp.ranked[:10], ['title', 'authors']]

In [ ]:
from IPython.display import Markdown
Markdown(open('../results/results.md').read())